In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [3]:
#Self-Attention Layer
class SelfAttention(nn.Module):
    def __init__(self, in_channels):
        super(SelfAttention, self).__init__()
        self.query = nn.Conv2d(in_channels, in_channels // 8, 1)
        self.key   = nn.Conv2d(in_channels, in_channels // 8, 1)
        self.value = nn.Conv2d(in_channels, in_channels, 1)
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        """
        x: (B, C, H, W)
        """
        B, C, H, W = x.size()
        query = self.query(x).view(B, -1, H * W).permute(0, 2, 1)   # (B, N, C')
        key   = self.key(x).view(B, -1, H * W)                      # (B, C', N)
        energy = torch.bmm(query, key)                              # (B, N, N)
        attention = F.softmax(energy, dim=-1)
        value = self.value(x).view(B, -1, H * W)                    # (B, C, N)
        out = torch.bmm(value, attention.permute(0, 2, 1))          # (B, C, N)
        out = out.view(B, C, H, W)
        out = self.gamma * out + x
        return out

In [4]:
#Cross-Attention Layer 
class CrossAttention(nn.Module):
    def __init__(self, img_channels, text_dim, heads=4):
        super(CrossAttention, self).__init__()
        self.heads = heads
        self.scale = math.sqrt(img_channels // heads)

        self.query = nn.Conv2d(img_channels, img_channels, 1)
        self.key   = nn.Linear(text_dim, img_channels)
        self.value = nn.Linear(text_dim, img_channels)
        self.output = nn.Conv2d(img_channels, img_channels, 1)

    def forward(self, img_feat, text_emb):
        """
        img_feat: (B, C, H, W)
        text_emb: (B, T, D)
        """
        B, C, H, W = img_feat.size()
        T = text_emb.size(1)

        q = self.query(img_feat).view(B, self.heads, C // self.heads, H * W).permute(0, 1, 3, 2)
        k = self.key(text_emb).view(B, T, self.heads, C // self.heads).permute(0, 2, 1, 3)
        v = self.value(text_emb).view(B, T, self.heads, C // self.heads).permute(0, 2, 1, 3)

        attn = torch.matmul(q, k.transpose(-2, -1)) / self.scale
        attn = F.softmax(attn, dim=-1)
        out = torch.matmul(attn, v)
        out = out.permute(0, 1, 3, 2).contiguous().view(B, C, H, W)
        return self.output(out) + img_feat  # residual connection


In [5]:
#Generator with Attention
class GenBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(GenBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="nearest"),
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(True),
        )

    def forward(self, x):
        return self.block(x)

class Generator(nn.Module):
    def __init__(self, z_dim=128, text_dim=256, base_ch=64):
        super(Generator, self).__init__()
        self.fc = nn.Linear(z_dim, base_ch * 8 * 4 * 4)
        self.block1 = GenBlock(base_ch * 8, base_ch * 4)
        self.block2 = GenBlock(base_ch * 4, base_ch * 2)
        self.self_attn = SelfAttention(base_ch * 2)
        self.cross_attn = CrossAttention(base_ch * 2, text_dim)
        self.block3 = GenBlock(base_ch * 2, base_ch)
        self.to_rgb = nn.Conv2d(base_ch, 3, 3, padding=1)

    def forward(self, z, text_emb):
        B = z.size(0)
        x = self.fc(z).view(B, -1, 4, 4)
        x = self.block1(x)
        x = self.block2(x)
        x = self.self_attn(x)
        x = self.cross_attn(x, text_emb)
        x = self.block3(x)
        img = torch.tanh(self.to_rgb(x))
        return img


In [8]:
#Discriminator with Attention
class DiscBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(DiscBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 4, stride=2, padding=1),
            nn.LeakyReLU(0.2, True),
        )

    def forward(self, x):
        return self.block(x)

class Discriminator(nn.Module):
    def __init__(self, text_dim=256, base_ch=64):
        super(Discriminator, self).__init__()
        self.block1 = DiscBlock(3, base_ch)
        self.block2 = DiscBlock(base_ch, base_ch * 2)
        self.self_attn = SelfAttention(base_ch * 2)
        self.block3 = DiscBlock(base_ch * 2, base_ch * 4)
        
        # After block3, spatial size = 4x4 (assuming 32x32 input)
        img_feat_dim = base_ch * 4 * 4 * 4  # 64 * 4 * 4 * 4 = 4096 for base_ch=64
        
        # Match projection dimensions
        self.img_proj = nn.Linear(img_feat_dim, text_dim)
        self.text_proj = nn.Linear(text_dim, text_dim)
        
        # Final score
        self.fc_out = nn.Linear(text_dim, 1)

    def forward(self, img, text_emb):
        x = self.block1(img)
        x = self.block2(x)
        x = self.self_attn(x)
        x = self.block3(x)
        x = x.view(x.size(0), -1)  # Flatten image features
        
        # Average token embeddings (B, T, D) → (B, D)
        t = text_emb.mean(dim=1)
        
        # Project both to same space
        x_proj = self.img_proj(x)
        t_proj = self.text_proj(t)
        
        # Combine via elementwise product
        combined = x_proj * t_proj
        out = self.fc_out(combined)
        return out


In [9]:
#Training Example 
if __name__ == "__main__":
    # Example input shapes
    B, z_dim, text_dim, T = 4, 128, 256, 10
    z = torch.randn(B, z_dim)
    text_emb = torch.randn(B, T, text_dim)

    # Instantiate models
    G = Generator(z_dim=z_dim, text_dim=text_dim)
    D = Discriminator(text_dim=text_dim)

    # Generate fake images
    fake_imgs = G(z, text_emb)
    print("Generated images shape:", fake_imgs.shape)  # (B, 3, 32, 32)

    # Discriminator output
    out = D(fake_imgs, text_emb)
    print("Discriminator output shape:", out.shape)  # (B, 1)

Generated images shape: torch.Size([4, 3, 32, 32])
Discriminator output shape: torch.Size([4, 1])
